### Leer todos los datos requeridos

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date =dbutils.widgets.get("p_file_date")

In [0]:
# movies_df = spark.read.parquet(f"{silver_folder_path}/movies")

# ahora lee los datos dentro de la tabla... no de los archivos:
movies_df = spark.read.table("movie_silver.movies") \
                    .filter(f"file_date = '{v_file_date}'")

In [0]:
# languages_df = spark.read.parquet(f"{silver_folder_path}/languages")
languages_df = spark.read.table("movie_silver.languages")

In [0]:
# movies_language_df = spark.read.parquet(f"{silver_folder_path}/movies_languages")

movies_language_df = spark.read.table("movie_silver.movies_languages") \
                        .filter(f"file_date = '{v_file_date}'")

In [0]:
# genres_df = spark.read.parquet(f"{silver_folder_path}/genre")
genres_df = spark.read.table("movie_silver.genre")


In [0]:
# movies_genres_df = spark.read.parquet(f"{silver_folder_path}/movies_genres")
movies_genres_df = spark.read.table("movie_silver.movies_genres") \
                        .filter(f"file_date = '{v_file_date}'")

#### Join "languages" y "movies_languages"

In [0]:
language_mov_lan_df = languages_df.join(movies_language_df, 
                                        movies_language_df.language_id == languages_df.language_id, 
                                        "inner") \
                                    .select(languages_df.language_name, movies_language_df.language_id, movies_language_df.movie_id)

#### Join "genres" y "movies_genres"

In [0]:
genres_mov_gen_df = genres_df.join(movies_genres_df,
                                    movies_genres_df.genre_id == genres_df.genre_id,
                                    "inner") \
                                .select(genres_df.genre_name, genres_df.genre_id, movies_genres_df.movie_id)

#### Join "movies_df", "languages_mov_lan_df" y "genres_mov_lan_df"

##### - Filtrar las películas donde su fecha de lanzamiento sea mayor o igual a 2000

In [0]:
movies_filter_df = movies_df.filter("year_release_date >= 2000")

In [0]:
results_movies_genres_languages_df = movies_filter_df.join(language_mov_lan_df,
                                            movies_filter_df.movie_id == language_mov_lan_df.movie_id,
                                            "inner") \
                                                       .join(genres_mov_gen_df,
                                                             movies_filter_df.movie_id == genres_mov_gen_df.movie_id,
                                                             "inner")

##### - Agregar la columna "created_date"

In [0]:
from pyspark.sql.functions import lit

In [0]:
results_df = results_movies_genres_languages_df \
                .select(movies_filter_df.movie_id, "language_id", "genre_id","title", "duration_time", "release_date", "vote_average","language_name", "genre_name") \
                .withColumn("created_date", lit(v_file_date))

In [0]:
display(results_df)

##### - Ordernar por la columna "release_date" de manera descendente

In [0]:
results_order_by_df = results_df.orderBy(results_df.release_date.desc())

In [0]:
display(results_order_by_df)

### Escribir los datos en el datalake en formato "Parquet"

In [0]:
# hago una validacion con merge

merge_condition = 'tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.genre_id = src.genre_id AND tgt.created_date=src.created_date'
merge_delta_lake(results_order_by_df, "movie_gold", "results_movie_genre_language", merge_condition, "created_date")

In [0]:
# display(spark.read.parquet(f"{gold_folder_path}/results_movie_genre_language"))

In [0]:
%sql
SELECT * FROM movie_gold.results_movie_genre_language